# BERT-CNN Training

Trains a `ConvDenoiser` with BERT-style masked-position pretraining.

During each training step, `mask_rate` (default 15%) of sequence positions have
their input probability vector replaced with uniform (1/20). The model must
predict the correct amino acid at those positions using only neighbouring context.
Loss is computed on **all** positions — masked positions force context learning,
unmasked positions maintain the standard denoising objective.

Eval is identical to the standard unified eval (no masking applied).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import torch
import torch.nn.functional as F
import h5py
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm import tqdm

from data.vibeseq.simulation.naive import load_confusion_matrix
from models.properties import AA_ORDER
from models.conv_denoiser import ConvDenoiser
from models.unified_dataset import (
    PERTURBATION_CONDITIONS,
    make_train_loader, make_eval_loader,
    apply_bert_mask,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SEED      = 42
EPOCHS    = 30
BATCH     = 32
LR        = 1e-3
MAX_LEN   = 1000
MILD_EPS  = 0.02
MASK_RATE = 0.15        # fraction of positions to mask during training
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

SRC_H5    = Path('../data/synthetic/naive_full_seed42.h5')
EVAL_H5   = Path('../data/synthetic/eval_unified_seed42.h5')
CKPT_PATH = Path('../models/checkpoints/unified_cnn_bert.pt')
CM_PATH   = Path('../assets/rf_cm.npy')

aa_to_idx = {aa: i for i, aa in enumerate(AA_ORDER)}
classes, cm_frac = load_confusion_matrix(CM_PATH)

print(f'Device:    {DEVICE}')
print(f'Mask rate: {MASK_RATE:.0%}')

In [ ]:
with h5py.File(SRC_H5, 'r') as f:
    all_pids  = list(f.keys())
    sequences = {pid: f[pid]['sequence'][()].decode() for pid in all_pids}

sequences = {pid: seq for pid, seq in sequences.items() if len(seq) <= MAX_LEN}
all_pids  = list(sequences.keys())

rng_split  = np.random.default_rng(SEED)
pids       = rng_split.permutation(all_pids).tolist()
split      = int(len(pids) * 0.8)
train_pids = pids[:split]
val_pids   = pids[split:]

train_sequences = [(pid, sequences[pid]) for pid in train_pids]

print(f'Proteins (len ≤ {MAX_LEN}): {len(all_pids):,}')
print(f'Train: {len(train_pids):,}   Val: {len(val_pids):,}')

In [ ]:
train_loader = make_train_loader(
    train_sequences, classes, cm_frac, aa_to_idx,
    batch_size=BATCH, num_workers=4, mild_eps=MILD_EPS, max_len=MAX_LEN,
)

eval_loaders = {
    cond: make_eval_loader(EVAL_H5, cond, val_pids, aa_to_idx,
                           batch_size=64, mild_eps=MILD_EPS)
    for cond in PERTURBATION_CONDITIONS
}

print(f'Train batches/epoch: {len(train_loader):,}')
print(f'Eval conditions:     {len(eval_loaders)}')

In [ ]:
model = ConvDenoiser(d=128, dropout=0.1).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

model.fit_stats(train_loader, device=DEVICE)
print('fit_stats done')

In [ ]:
def run_epoch(model, loader, optimizer=None, device='cpu', mask_rate=0.0):
    """Single train or eval pass.

    mask_rate > 0 applies BERT-style masking to the input during training.
    Eval always runs with mask_rate=0 (no masking).
    """
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = total_correct = total_tokens = 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for probs, labels in loader:
            probs, labels = probs.to(device), labels.to(device)

            if training and mask_rate > 0:
                probs, _ = apply_bert_mask(probs, mask_rate)

            logits = model(probs)
            loss = F.cross_entropy(
                logits.reshape(-1, 20), labels.reshape(-1), ignore_index=-1
            )
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            mask = labels != -1
            total_correct += (logits.argmax(-1)[mask] == labels[mask]).sum().item()
            total_tokens  += mask.sum().item()
            total_loss    += loss.item() * mask.sum().item()

    return total_loss / total_tokens, total_correct / total_tokens


@torch.no_grad()
def eval_all_conditions(model, eval_loaders, device='cpu'):
    model.eval()
    return {
        cond: run_epoch(model, loader, optimizer=None, device=device)[1]
        for cond, loader in eval_loaders.items()
    }

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
best_avg_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, DEVICE,
                                mask_rate=MASK_RATE)
    scheduler.step()

    if epoch % 5 == 0 or epoch == EPOCHS:
        cond_accs = eval_all_conditions(model, eval_loaders, DEVICE)
        avg_acc   = np.mean(list(cond_accs.values()))
        history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc,
                        'avg_acc': avg_acc, **cond_accs})

        if avg_acc > best_avg_acc:
            best_avg_acc = avg_acc
            torch.save(model.state_dict(), CKPT_PATH)

        print(f'Epoch {epoch:3d}  '
              f'tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.3%}  '
              f'avg_val={avg_acc:.3%}  '
              f'gamma1={cond_accs["perturb_gamma1"]:.3%}  '
              f'scr20={cond_accs["scramble_20pct"]:.3%}  '
              f'lc20={cond_accs["lowconf_20pct"]:.3%}')
    else:
        history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc})
        print(f'Epoch {epoch:3d}  tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.3%}')

print(f'\nBest avg val acc: {best_avg_acc:.3%}  →  {CKPT_PATH}')

In [ ]:
LABELS = {
    'ground_truth':    'Ground truth',
    'perturb_gamma50': 'Dirichlet γ=50',
    'perturb_gamma10': 'Dirichlet γ=10',
    'perturb_gamma3':  'Dirichlet γ=3',
    'perturb_gamma1':  'Dirichlet γ=1',
    'scramble_1pct':   'Scramble 1%',
    'scramble_5pct':   'Scramble 5%',
    'scramble_10pct':  'Scramble 10%',
    'scramble_20pct':  'Scramble 20%',
    'lowconf_1pct':    'Low-conf 1%',
    'lowconf_5pct':    'Low-conf 5%',
    'lowconf_10pct':   'Low-conf 10%',
    'lowconf_20pct':   'Low-conf 20%',
}

model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
final_accs = eval_all_conditions(model, eval_loaders, DEVICE)

@torch.no_grad()
def argmax_baseline(loader):
    correct = total = 0
    for probs, labels in loader:
        mask = labels != -1
        correct += (probs.argmax(-1)[mask] == labels[mask]).sum().item()
        total   += mask.sum().item()
    return correct / total

records = []
for cond in PERTURBATION_CONDITIONS:
    argmax_acc = argmax_baseline(eval_loaders[cond])
    records.append({
        'condition': LABELS[cond],
        'argmax':    argmax_acc,
        'bert_cnn':  final_accs[cond],
        'delta':     final_accs[cond] - argmax_acc,
    })

df = pd.DataFrame(records)
df_display = df.copy()
df_display['argmax']   = df['argmax'].map('{:.3%}'.format)
df_display['bert_cnn'] = df['bert_cnn'].map('{:.3%}'.format)
df_display['delta']    = df['delta'].map('{:+.3%}'.format)
df_display

In [ ]:
eval_hist = [h for h in history if 'avg_acc' in h]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot([h['epoch'] for h in history], [h['tr_loss'] for h in history], 'o-')
axes[0].set(xlabel='Epoch', ylabel='Train loss', title='Training loss (with BERT masking)')

axes[1].plot([h['epoch'] for h in eval_hist], [h['avg_acc'] for h in eval_hist],
             'o-', label='Avg across conditions')
for cond in ['perturb_gamma1', 'scramble_20pct', 'lowconf_20pct']:
    if cond in eval_hist[0]:
        axes[1].plot([h['epoch'] for h in eval_hist], [h[cond] for h in eval_hist],
                     '--', label=LABELS[cond], alpha=0.7)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=1))
axes[1].set(xlabel='Epoch', ylabel='Val accuracy', title='Validation accuracy (no masking)')
axes[1].legend(fontsize=8)

fig.suptitle(f'BERT-CNN  ·  mask_rate={MASK_RATE:.0%}', fontsize=12)
fig.tight_layout()
plt.show()